In [ ]:
# Install compatible packages for this RAG notebook.
!pip install -q -U \
    "langchain>=1.0,<2.0" \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    chromadb \
    sentence-transformers \
    "transformers==4.57.1" \
    accelerate \
    gradio

# Run this only if you see a torchvision / PreTrainedModel import error:
!pip uninstall -y -q torchvision


In [ ]:
# Core imports and version check
import os
import shutil
import torch

import langchain
import transformers
import chromadb

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

print("LangChain version:", langchain.__version__)
print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

LangChain version: 1.3.1
Transformers version: 4.57.1
Torch version: 2.10.0+cu128
CUDA available: True


/tmp/ipykernel_1112/1966785336.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [ ]:
# Step 1: Load and chunk text using LangChain

documents = [
    """Space travel can have profound effects on the human body. Astronauts experience muscle atrophy, bone density loss, and fluid redistribution due to the microgravity environment. Long-term exposure to cosmic radiation is also a significant concern.""",
    """NASA uses various techniques including exercise, diet, and medication to mitigate the health effects of space travel. Continuous monitoring and research aboard the International Space Station (ISS) help scientists understand these challenges."""
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = []
for doc in documents:
    chunks.extend(text_splitter.split_text(doc))

print(f"Created {len(chunks)} chunks")
for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i}:\n{chunk}")

Created 2 chunks

Chunk 1:
Space travel can have profound effects on the human body. Astronauts experience muscle atrophy, bone density loss, and fluid redistribution due to the microgravity environment. Long-term exposure to cosmic radiation is also a significant concern.

Chunk 2:
NASA uses various techniques including exercise, diet, and medication to mitigate the health effects of space travel. Continuous monitoring and research aboard the International Space Station (ISS) help scientists understand these challenges.


In [ ]:
# Step 2: Create the embedding model

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [ ]:
# Step 3: Create a Chroma vector store

docs = [Document(page_content=chunk) for chunk in chunks]

# Works in both Google Colab and local Jupyter.
persist_dir = "/content/chroma_db" if os.path.exists("/content") else "./chroma_db"

if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=persist_dir,
    collection_name="space_health_documents"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Vector store created at: {persist_dir}")

Vector store created at: /content/chroma_db


In [ ]:
# Step 4: Inspect the first chunk embedding
first_chunk = chunks[0]
print("Text of first chunk:\n", first_chunk)

first_embedding = embedding_model.embed_query(first_chunk)

print("\nEmbedding vector first 10 values:\n", first_embedding[:10])
print("\nEmbedding vector length:", len(first_embedding))

Text of first chunk:
 Space travel can have profound effects on the human body. Astronauts experience muscle atrophy, bone density loss, and fluid redistribution due to the microgravity environment. Long-term exposure to cosmic radiation is also a significant concern.

Embedding vector first 10 values:
 [0.06894783675670624, 0.008361603133380413, 0.10940364748239517, 0.07590483129024506, -0.011964731849730015, 0.022983526811003685, 0.07899022102355957, 0.02986052632331848, -0.03302625194191933, 0.022847749292850494]

Embedding vector length: 384


In [ ]:
# Step 5: Load the Hugging Face LLM

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Disable KV cache to avoid DynamicCache/seen_tokens compatibility errors.
model.config.use_cache = False
model.generation_config.use_cache = False

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
    use_cache=False,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

print("LLM loaded successfully:", model_name)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

... (5 outputs left out)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM loaded successfully: microsoft/Phi-3-mini-4k-instruct


In [ ]:
# Step 6: Build the RAG chain using the current LangChain LCEL style
custom_prompt = PromptTemplate.from_template(
    """You are an expert AI assistant. Use only the provided context to answer the user's question.
If the answer is not in the context, say that the context does not provide enough information.

Context:
{context}

Question:
{question}

Answer:"""
)

def format_docs(retrieved_docs):
    return "\n\n---\n\n".join(doc.page_content for doc in retrieved_docs)

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully.")

RAG chain created successfully.


In [ ]:
# Step 7: Define the query function
def clean_answer(answer):
    """Remove prompt spillover if the model repeats sections of the prompt."""
    answer = answer.strip()

    if "Answer:" in answer:
        answer = answer.split("Answer:")[-1].strip()

    cutoff_phrases = ["Context:", "Question:", "\n---"]
    for phrase in cutoff_phrases:
        if phrase in answer:
            answer = answer.split(phrase)[0].strip()

    return answer

def answer_query(query):
    answer = rag_chain.invoke(query)
    answer = clean_answer(answer)

    relevant_chunks = vectorstore.similarity_search(query, k=2)
    context_text = "\n\n---\n\n".join(doc.page_content for doc in relevant_chunks)

    return f"""**Answer:**

{answer}

**Relevant Context:**

{context_text}"""

# Quick test
print(answer_query("What health problems can astronauts experience during space travel?"))

**Answer:**

The Hubble Space Telescope has contributed to

**Relevant Context:**

Space travel can have profound effects on the human body. Astronauts experience muscle atrophy, bone density loss, and fluid redistribution due to the microgravity environment. Long-term exposure to cosmic radiation is also a significant concern.

---

NASA uses various techniques including exercise, diet, and medication to mitigate the health effects of space travel. Continuous monitoring and research aboard the International Space Station (ISS) help scientists understand these challenges.


In [ ]:
# Step 8: Launch the Gradio interface in Google Colab
import gradio as gr

interface = gr.Interface(
    fn=answer_query,
    inputs=gr.Textbox(label="Ask a question"),
    outputs=gr.Markdown(label="Answer"),
    title="RAG with Phi-3 and Chroma",
    description="Ask questions about the sample space-travel documents."
)

interface.launch(
    share=True,
    debug=False,
    inline=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b20d36a2b6f9b5568a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
